# 可选实验 - 神经元与层
在本实验中，我们将探索神经元/单元和层的内部工作原理。具体而言，本实验会将它们与课程 1 中你已经掌握的模型——回归/线性模型和逻辑回归模型——进行类比。本实验还会介绍 TensorFlow，并演示如何在该框架中实现这些模型。
<figure>
   <img src="../work/images/C2_W1_NeuronsAndLayers.png"  style="width:540px;height:200px;" >
</figure>

## 软件包
**TensorFlow 和 Keras**  
TensorFlow 是 Google 开发的机器学习软件包。2019 年，Google 将 Keras 集成到 TensorFlow 中，并发布了 TensorFlow 2.0。Keras 是 François Chollet 独立开发的框架，为 TensorFlow 提供了一个简单的、以层为中心的接口。本课程将使用 Keras 接口。

In [1]:
%pip install --upgrade pip

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
  Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.1.1
    Uninstalling pip-26.1.1:
      Successfully uninstalled pip-26.1.1
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install tensorflow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras import Sequential
from tensorflow.keras.losses import MeanSquaredError, BinaryCrossentropy
from tensorflow.keras.activations import sigmoid
from lab_utils_common import dlc
from lab_neurons_utils import plt_prob_1d, sigmoidnp, plt_linear, plt_logistic
plt.style.use('./deeplearning.mplstyle')
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## 无激活函数的神经元 - 回归/线性模型

### 数据集
我们将使用课程 1 中的一个示例：房价线性回归。

In [ ]:
X_train = np.array([[1.0], [2.0]], dtype=np.float32)           #(size in 1000 square feet)
Y_train = np.array([[300.0], [500.0]], dtype=np.float32)       #(price in 1000s of dollars)

fig, ax = plt.subplots(1,1)
ax.scatter(X_train, Y_train, marker='x', c='r', label="Data Points")
ax.legend( fontsize='xx-large')
ax.set_ylabel('Price (in 1000s of dollars)', fontsize='xx-large')
ax.set_xlabel('Size (1000 sqft)', fontsize='xx-large')
plt.show()

### 回归/线性模型
无激活函数的神经元所实现的函数与课程 1 中的线性回归相同：
$$ f_{\mathbf{w},b}(x^{(i)}) = \mathbf{w}\cdot x^{(i)} + b \tag{1}$$


我们可以定义一个只含一个神经元或单元的层，并将其与熟悉的线性回归函数进行比较。

In [ ]:
linear_layer = tf.keras.layers.Dense(units=1, activation = 'linear', )

让我们检查一下权重。

In [ ]:
linear_layer.get_weights()

目前还没有权重，因为权重尚未实例化。让我们在 `X_train` 中的一个样本上试运行模型，这会触发权重实例化。请注意，层的输入必须是二维的，因此需要重塑其形状。

In [ ]:
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)

结果是一个形状为 (1,1)、仅含一个条目的张量（数组的另一种称呼）。
现在来查看权重和偏置。这些权重会被随机初始化为较小的数，而偏置默认初始化为零。

In [ ]:
w, b= linear_layer.get_weights()
print(f"w = {w}, b={b}")

只有一个输入特征的线性回归模型 (1) 将包含一个权重和一个偏置。这与上面的 `linear_layer` 维度相符。

权重会初始化为随机值，因此让我们将它们设置为一些已知值。

In [ ]:
set_w = np.array([[200]])
set_b = np.array([100])

# set_weights takes a list of numpy arrays
linear_layer.set_weights([set_w, set_b])
print(linear_layer.get_weights())

让我们将方程 (1) 与该层的输出进行比较。

In [ ]:
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)
alin = np.dot(set_w,X_train[0].reshape(1,1)) + set_b
print(alin)

它们产生了相同的值！
现在，可以使用线性层对训练数据进行预测。

In [ ]:
prediction_tf = linear_layer(X_train)
prediction_np = np.dot( X_train, set_w) + set_b

In [ ]:
plt_linear(X_train, Y_train, prediction_tf, prediction_np)

## 使用 Sigmoid 激活函数的神经元
使用 sigmoid 激活函数的神经元/单元所实现的函数，与课程 1 中的逻辑回归相同：
$$ f_{\mathbf{w},b}(x^{(i)}) = g(\mathbf{w}x^{(i)} + b) \tag{2}$$
其中，$$g(x) = sigmoid(x)$$

让我们将 $w$ 和 $b$ 设置为一些已知值，并检查模型。

### 数据集
我们将使用课程 1 中的逻辑回归示例。

In [ ]:
X_train = np.array([0., 1, 2, 3, 4, 5], dtype=np.float32).reshape(-1,1)  # 2-D Matrix
Y_train = np.array([0,  0, 0, 1, 1, 1], dtype=np.float32).reshape(-1,1)  # 2-D Matrix

In [ ]:
pos = Y_train == 1
neg = Y_train == 0
X_train[pos]

In [ ]:
pos = Y_train == 1
neg = Y_train == 0

fig,ax = plt.subplots(1,1,figsize=(4,3))
ax.scatter(X_train[pos], Y_train[pos], marker='x', s=80, c = 'red', label="y=1")
ax.scatter(X_train[neg], Y_train[neg], marker='o', s=100, label="y=0", facecolors='none', 
              edgecolors=dlc["dlblue"],lw=3)

ax.set_ylim(-0.08,1.1)
ax.set_ylabel('y', fontsize=12)
ax.set_xlabel('x', fontsize=12)
ax.set_title('one variable plot')
ax.legend(fontsize=12)
plt.show()

### 逻辑神经元
可以通过添加 sigmoid 激活函数来实现“逻辑神经元”。此时，该神经元的函数由上面的公式 (2) 描述。
本节将创建一个包含逻辑层的 TensorFlow 模型，以演示另一种创建模型的方法。TensorFlow 最常用于创建多层模型。[Sequential](https://keras.io/guides/sequential_model/) 模型是构建这些模型的一种便捷方式。

In [ ]:
model = Sequential(
    [
        tf.keras.layers.Dense(1, input_dim=1,  activation = 'sigmoid', name='L1')
    ]
)

`model.summary()` 显示了模型中的层及参数数量。该模型只有一层，而且这一层只有一个单元。该单元包含两个参数：$w$ 和 $b$。

In [ ]:
model.summary()

In [ ]:
logistic_layer = model.get_layer('L1')
w,b = logistic_layer.get_weights()
print(w,b)
print(w.shape,b.shape)

让我们将权重和偏置设置为一些已知值。

In [ ]:
set_w = np.array([[2]])
set_b = np.array([-4.5])
# set_weights takes a list of numpy arrays
logistic_layer.set_weights([set_w, set_b])
print(logistic_layer.get_weights())

让我们将公式 (2) 与该层的输出进行比较。

In [ ]:
a1 = model.predict(X_train[0].reshape(1,1))
print(a1)
alog = sigmoidnp(np.dot(set_w,X_train[0].reshape(1,1)) + set_b)
print(alog)

它们产生相同的值！
现在，可以使用逻辑层和 NumPy 模型对训练数据进行预测。

In [ ]:
plt_logistic(X_train, Y_train, model, set_w, set_b, pos, neg)

上面的阴影反映了 sigmoid 的输出，其取值范围为 0 到 1。

# 恭喜！
你构建了一个非常简单的神经网络，并探索了神经元与课程 1 中线性回归和逻辑回归之间的相似之处。